# Final Project

- 驗證正確性和效能
- 跟既有的 Solver 比較
- Must support parallelization
- Must use GitHub for development

# Lattice Boltzmann Method (LBM)

- Fluids can be imagined as consisting of a large number of small particles undergoing random motion.
- The exchange of momentum and energy is achieved via particle **streaming** and **collision**.
- The LBM simplifies Boltzmann's idea of gas dynamics by reducing the number of particles and confining them to the nodes of a lattice.
- In actual implementations, **streaming** and **collision** are computed separately.

The process can be modelled by the Boltzmann transport equation
$$
\frac{\partial f(\vec{x},t)}{\partial t} + \vec{u} \cdot \nabla f(\vec{x}, t) = \Omega,
$$
where

- $f(\vec{x},t)$ is the particle distribution function.

- $\vec{u}$ is the particle velocity.

- $\Omega$ is the collision operator.

For a two-dimensional model, a particle is restricted to stream in 9 possible directions. A typical lattice node of the D2Q9 model with 9 velocities $\vec{e}_i$ is defined by
$$
\vec{e}_i = 
\begin{cases}
(0, 0) &\text{if }i = 0 \\
(1, 0), (0, 1), (-1, 0), (0, -1) &\text{if }i = 1, 2, 3, 4 \\
(1, 1), (-1, 1), (-1, -1), (1, -1) &\text{if }i = 5, 6, 7, 8
\end{cases}.
$$
We can associate a discrete particle distribution function with
$$
f_i(\vec{x}, t),\;i = 0, \dots, 8,
$$
which describes the probability of **streaming** in one particular direction.

The *macroscopic fluid density* can be defined as the sum of microscopic particle distribution functions
$$
\rho(\vec{x}, t) = \sum_{i = 0}^8f_i(\vec{x}, t).
$$
The *macroscopic velocity* is the weighted (by the distribution functions $f_i$) average of microscopic velocities $\vec{e}_i$
$$
\vec{u}(\vec{x}, t) = \frac{1}{\rho}\sum_{i = 0}^8cf_i\vec{e}_i.
$$
The **streaming** and **collision** processes are given by
$$
f_i(\vec{x} + c\vec{e}_i\Delta t, t + \Delta t) - f_i(\vec{x}, t) = -\frac{[f_i(\vec{x}, t) - f_i^{\text{eq}}(\vec{x}, t)]}{\tau},
$$
where

- LHS represents **streaming**, and RHS describes **collision**.

- $f_i^{\text{eq}}(\vec{x}, t)$ is the equilibrium distribution.

- $\tau$ is the relaxation time towards local equilibrium.

The Bhatnagar-Gross-Krook (BGK) collision model is used to describe single-phase flows
$$
f_i^{\text{eq}}(\vec{x}, t) = w_i\rho + \rho s_i\left[\vec{u}(\vec{x}, t)\right],
$$
where $s_i\left(\vec{u}\right)$ is defined as
$$
s_i\left[\vec{u}(\vec{x}, t)\right] = w_i\left[3\frac{\vec{e}_i \cdot \vec{u}}{c} + \frac{9}{2}\frac{(\vec{e}_i \cdot \vec{u})^2}{c^2} - \frac{3}{2}\frac{\vec{u} \cdot \vec{u}}{c^2}\right],
$$
$w_i$ are the weights
$$
w_i = 
\begin{cases}
\frac{4}{9} &\text{if }i = 0 \\
\frac{1}{9} &\text{if }i = 1, 2, 3, 4 \\
\frac{1}{36} &\text{if }i = 5, 6, 7, 8
\end{cases}.
$$
and $c = \frac{\Delta x}{\Delta t}$ is the lattice speed. The fluid kinematic viscosity $\nu$ in the D2Q9 model is related to the relaxation time $\tau$ by
$$
\nu = \frac{2\tau - 1}{6}\frac{(\Delta x)^2}{\Delta t}.
$$

The algorithm is summarized as

1. Initialize $\rho$, $\vec{u}$, $f_i$ and $f_i^{\text{eq}}$.

2. Streaming: Move $f_i \to f_i^*$ along the direction of $\vec{e}_i$.

3. Compute macroscopic $\rho$ and $\vec{u}$ from $f_i$.

4. Compute $f_i^{\text{eq}}$.

5. Collision: Calculate the updated distribution function $f_i$ with
   $$
   f_i = f_i^* - \frac{1}{\tau}\left(f_i^* - f_i^{\text{eq}}\right).
   $$

6. Repeat steps 2 to 5.

# Boundary Conditions (BCs)

## Bounce-back BCs

- When a fluid particle reaches a boundary node, the particle will scatter the fluid back along its incoming direction.
- There are two types of implementations: the on-grid and the mid-grid bounce-back.
  - The on-grid bounce-back is simple and computationally efficient, since this BC just reflects a fluid particle when it hits the wall.
  - The mid-grid bounce-back needs a fictitious node outside the wall to apply the reflection.
  - However, the on-grid bounce-back is only first-order accurate, and the mid-grid is second-order accurate.

## Zou-He BCs

- The main idea is to model flows with given velocity, pressure, or density at the boundary.
- This type of BC depends on the orientation of the boundary and is thus hard to generalize for complex geometries.

Rewriting the *macroscopic fluid density* and *velocity* equations, we have
$$
\begin{aligned}
f_1 + f_5 + f_8 &= \rho - (f_0 + f_2 + f_4 + f_3 + f_6 + f_7); \\
f_1 + f_5 + f_8 &= \rho u + (f_3 + f_6 + f_7); \\
f_5 - f_8 &= \rho v - f_2 + f_4 - f_6 + f_7.
\end{aligned}
$$
$\rho$ is thus
$$
\rho = \frac{1}{1 - u}\left[f_0 + f_2 + f_4 + 2(f_3 + f_6 + f_7)\right].
$$
To solve for $f_1$, $f_5$, and $f_8$, the assumption is that the bounce-back rule still holds for the non-equilibrium part of the particle distribution normal to the boundary. The fourth equation is given by
$$
f_1 - f_1^{\text{eq}} = f_3 - f_3^{\text{eq}}.
$$
Thus, $f_1$, $f_5$, and $f_8$ are
$$
\begin{aligned}
f_1 &= f_3 + \frac{2}{3}\rho v; \\
f_5 &= f_7 - \frac{1}{2}(f_2 - f_4) + \frac{1}{6}\rho u + \frac{1}{2}\rho v; \\
f_8 &= f_6 + \frac{1}{2}(f_2 - f_4) + \frac{1}{6}\rho u - \frac{1}{2}\rho v.
\end{aligned}
$$

# Topics

- Steady Plane Poiseuille Flow
- Lid-driven Cavity Flow
- Flow past a Circular Cylinder
- Rayleigh-Bénard Convection

# Connection with the Navier-Stokes Equations